## Detectando casos de COVID a partir de imagens de Tomografia


Esse é um projeto de pós graduação em inteligência artificial cujo o objetivo deste conjunto de dados é incentivar a pesquisa e o desenvolvimento de métodos de
inteligência artificial capazes de identificar se uma pessoa está infectada pelo SARS-CoV-2 por
meio da análise de suas tomografias computadorizadas.

O dataset em questão está disponível no kaggle e foi coletado de pacientes reais em hospitais
no estado de São Paulo.

Nesse projeto, as principais habilidades a serem exercitadas estão relacionadas ao
Processamento de Imagens e a utilização de modelos de Deep Learning para classificação

fonte de dados: https://www.kaggle.com/datasets/plameneduardo/sarscov2-ctscan-dataset

### Carregamento e Pré-processamento básico para normalizar imagens

In [44]:
# importando as blibliotecas
import os
from PIL import Image
import cv2
import numpy as np
from sklearn.model_selection import train_test_split

In [45]:
# Definindo o caminho para a pasta de imagens
covid_dir = r'C:\Users\geova\OneDrive\Ambiente de Trabalho\inteligencia artificial\5 machine learning\detectando casos de covid\COVID'
no_covid_dir = r'C:\Users\geova\OneDrive\Ambiente de Trabalho\inteligencia artificial\5 machine learning\detectando casos de covid\non-COVID'


In [46]:

# definindo um tamanho padrão pois quando tentei converter lá embaixo as listas de imagens para arrays, algumas imagens tinham tamanhos diferentes e isso dá erro no codigo
img_altura, img_largura = 150, 150

# lista para armazenar as imagens e labels das imagens que SÃO de COVID
imagens = []
labels = []


In [47]:
# Carregar imagens da pasta COVID
for filename in os.listdir(covid_dir):
    if filename.endswith('.png'): # verifica se no final do arquivo tem .png
        img_path = os.path.join(covid_dir, filename) # cria o caminho completo do arquivo
        try:
            # abre a imagem e converte para RGB porque algumas imagens tinham mais 3 de canais provavelmente RGBA e quanto tem o quarto canal que é o alpha, o PIL não consegue converter para array
            img = Image.open(img_path).convert('RGB')  # converte a imagem para RGB ou seja 3 canais
            img_redimensionada = img.resize((img_largura, img_altura))  # redimensiona a imagem para o tamanho padrão
            img_array = np.array(img_redimensionada)  # converte a imagem para um array numpy
            imagens.append(img_array)  # adiciona a imagem à lista de imagens
            labels.append(1)  # adiciona o label 1 para indicar que é COVID
        except Exception as e:
            print(f'Erro ao processar a imagem {filename}: {e}')

In [48]:
# Carregar imagens da pasta non-COVID
for filename in os.listdir(no_covid_dir):
    if filename.endswith('.png'):  # verifica se no final do arquivo tem .png
        img_path = os.path.join(no_covid_dir, filename)  # cria o caminho completo do arquivo
        try:
            img = Image.open(img_path).convert('RGB')  # abre a imagem e converte para RGB
            img_redimensionada = img.resize((img_largura, img_altura))  # redimensiona a imagem para o tamanho padrão
            img_array = np.array(img_redimensionada)  # converte a imagem para um array numpy
            imagens.append(img_array)  # adiciona a imagem à lista de imagens
            labels.append(0)  # adiciona a label 0 para indicar que não é COVID
        except Exception as e:
            print(f"Erro ao processar a imagem {filename}: {e}")

In [49]:

imagens = np.array(imagens)  # converte a lista de imagens para um array numpy
labels = np.array(labels)  # converte a lista de labels para um array numpy

imagens_normalizadas = imagens.astype('float32') / 255.0

print(f"Formato do array de imagens normalizadas: {imagens_normalizadas.shape}")
print(f"Valores dos pixels após a normalização (exemplo - primeiro pixel da primeira imagem): {imagens_normalizadas[0][0][0]}")

Formato do array de imagens normalizadas: (2481, 150, 150, 3)
Valores dos pixels após a normalização (exemplo - primeiro pixel da primeira imagem): [0.7529412 0.7529412 0.7529412]


In [50]:

# Dividir os dados em conjuntos de treinamento, validação e teste
X_train, X_temp, y_train, y_temp = train_test_split(imagens_normalizadas, labels, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Tamanho do conjunto de treinamento: {len(X_train)}")
print(f"Tamanho do conjunto de validação: {len(X_val)}")
print(f"Tamanho do conjunto de teste: {len(X_test)}")


Tamanho do conjunto de treinamento: 1984
Tamanho do conjunto de validação: 248
Tamanho do conjunto de teste: 249


In [51]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Definindo o formato de entrada das imagens (altura, largura, canais)
input_shape = (150, 150, 3) # Usamos 3 canais porque convertemos as imagens para RGB

# Criando o modelo
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid') # Camada de saída com 1 neurônio e função de ativação sigmoid para classificação binária (0 ou 1)
])

# Exibindo um resumo do modelo para ver sua arquitetura
model.summary()

# Compilando o modelo
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Treinando o modelo
epochs = 10 # Você pode ajustar o número de épocas conforme necessário

history = model.fit(X_train, y_train,
                    epochs=epochs,
                    validation_data=(X_val, y_val))

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_12 (Conv2D)              │ (None, 148, 148, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 72, 72, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_13 (MaxPooling2D) │ (None, 36, 36, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 34, 34, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_14 (MaxPooling2D) │ (None, 17, 17, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 36992)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 128)            │     4,735,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,828,481 (18.42 MB)

 Trainable params: 4,828,481 (18.42 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 14s 202ms/step - accuracy: 0.5037 - loss: 0.7723 - val_accuracy: 0.6573 - val_loss: 0.6460
Epoch 2/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 13s 206ms/step - accuracy: 0.6702 - loss: 0.5954 - val_accuracy: 0.8145 - val_loss: 0.4793
Epoch 3/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 12s 194ms/step - accuracy: 0.8069 - loss: 0.4369 - val_accuracy: 0.8306 - val_loss: 0.4121
Epoch 4/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 12s 194ms/step - accuracy: 0.8241 - loss: 0.3742 - val_accuracy: 0.8710 - val_loss: 0.3437
Epoch 5/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 12s 197ms/step - accuracy: 0.8728 - loss: 0.3049 - val_accuracy: 0.8266 - val_loss: 0.3683
Epoch 6/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 11s 185ms/step - accuracy: 0.8667 - loss: 0.3106 - val_accuracy: 0.8911 - val_loss: 0.3099
Epoch 7/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 12s 199ms/step - accuracy: 0.9196 - loss: 0.2063 - val_accuracy: 0.8831 - val_loss: 0.3768
Epoch 8/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 12s 196ms/step - accuracy: 0.9259 - loss: 0.1964 - val_accu

In [52]:
# Avaliando o modelo no conjunto de teste
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)

print(f'Perda (Loss) no conjunto de teste: {loss:.4f}')
print(f'Acurácia (Accuracy) no conjunto de teste: {accuracy:.4f}')

Perda (Loss) no conjunto de teste: 0.2400
Acurácia (Accuracy) no conjunto de teste: 0.8956


In [53]:
model.save('modelo_covid.h5')